In [1]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


In [2]:
# Make sure that the working directory is the project root.
%cd -q ../

In [3]:

import torch
import pickle
import numpy as np
from autoregltl import ted
from autoregltl.ltl import trace_check

from tqdm.auto import tqdm
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 192
mpl.rcParams['svg.fonttype'] = 'none'  # Critical: don't convert text to paths
plt.rcParams['pdf.use14corefonts'] = True

device = torch.device('cuda')

In [4]:
proposed_model_path = "models-scc/reverse-prop-epat/s1-noeda-42"
baseline_model_path = "../mare/models-prop/5ap/0000-rop-bn0-fn1-ada1-s42"

In [5]:
import json
import os

def compare_models(model1_path, model2_path, model1_name="Model 1", model2_name="Model 2"):
    """
    Compare configs and pytorch parameters between two models.
    
    Args:
        model1_path: Path to first model directory
        model2_path: Path to second model directory
        model1_name: Display name for first model
        model2_name: Display name for second model
    """
    # Load configs
    with open(os.path.join(model1_path, "config.json")) as f:
        config1 = json.load(f)

    with open(os.path.join(model2_path, "config.json")) as f:
        config2 = json.load(f)

    # Compare config keys
    keys1 = set(config1.keys())
    keys2 = set(config2.keys())

    print("=" * 60)
    print("CONFIG COMPARISON")
    print("=" * 60)

    only_in_model1_cfg = keys1 - keys2
    only_in_model2_cfg = keys2 - keys1
    common_keys_cfg = keys1 & keys2

    if only_in_model1_cfg:
        print(f"\n✓ Only in {model1_name} config: {sorted(only_in_model1_cfg)}")
    else:
        print(f"\n✓ No keys only in {model1_name} config")

    if only_in_model2_cfg:
        print(f"\n✗ Only in {model2_name} config: {sorted(only_in_model2_cfg)}")
    else:
        print(f"\n✗ No keys only in {model2_name} config")

    print(f"\n✓ Common keys ({len(common_keys_cfg)}): {sorted(common_keys_cfg)}")

    # Load pytorch models
    state1 = torch.load(os.path.join(model1_path, "pytorch_model.bin"), map_location='cpu')
    state2 = torch.load(os.path.join(model2_path, "pytorch_model.bin"), map_location='cpu')

    # Compare parameter keys
    params1 = set(state1.keys())
    params2 = set(state2.keys())

    print("\n" + "=" * 60)
    print("PYTORCH MODEL PARAMETER COMPARISON")
    print("=" * 60)

    only_in_model1_params = params1 - params2
    only_in_model2_params = params2 - params1
    common_params = params1 & params2

    if only_in_model1_params:
        print(f"\n✓ Only in {model1_name} model ({len(only_in_model1_params)}):")
        for param in sorted(only_in_model1_params):
            print(f"  - {param}: {state1[param].shape}")
    else:
        print(f"\n✓ No parameters only in {model1_name} model")

    if only_in_model2_params:
        print(f"\n✗ Only in {model2_name} model ({len(only_in_model2_params)}):")
        for param in sorted(only_in_model2_params):
            print(f"  - {param}: {state2[param].shape}")
    else:
        print(f"\n✗ No parameters only in {model2_name} model")

    print(f"\n✓ Common parameters ({len(common_params)}):")
    for param in sorted(list(common_params)[:5]):  # Show first 5
        print(f"  - {param}: {state1[param].shape}")
    if len(common_params) > 5:
        print(f"  ... and {len(common_params) - 5} more")
    
    # Compare merged_embedder.base_weight sizes
    print("\n" + "=" * 60)
    print("MERGED_EMBEDDER.BASE_WEIGHT COMPARISON")
    print("=" * 60)
    if "merged_embedder.base_weight" in state1 and "merged_embedder.base_weight" in state2:
        size1 = state1["merged_embedder.base_weight"].shape
        size2 = state2["merged_embedder.base_weight"].shape
        print(f"\n{model1_name} merged_embedder.base_weight size: {size1}")
        print(f"{model2_name} merged_embedder.base_weight size: {size2}")
        if size1 == size2:
            print("✓ Sizes match")
        else:
            print("✗ Sizes differ")
    else:
        print("\n✗ merged_embedder.base_weight not found in one or both models")

# Run initial comparison
print("INITIAL COMPARISON: Proposed vs Baseline\n")
compare_models(proposed_model_path, baseline_model_path, "Proposed", "Baseline")


INITIAL COMPARISON: Proposed vs Baseline

CONFIG COMPARISON

✓ Only in Proposed config: ['cross_attn', 'no_dec_agg', 'no_enc_agg']

✗ No keys only in Baseline config

✓ Common keys (17): ['d_embed_dec', 'd_embed_enc', 'd_ff', 'datatype', 'dec_pe', 'dropout', 'enc_pe', 'ff_activation', 'layer_norm_eps', 'max_decode_length', 'max_encode_length', 'merged_embedder', 'no_pe_cross_keys', 'num_heads', 'num_layers', 'tree_pos_enc', 'vocab']

PYTORCH MODEL PARAMETER COMPARISON

✓ Only in Proposed model (60):
  - decoder_stack.dec_layers.0.cross_attns.0.attn.K.bias: torch.Size([96])
  - decoder_stack.dec_layers.0.cross_attns.0.attn.K.weight: torch.Size([96, 96])
  - decoder_stack.dec_layers.0.cross_attns.0.attn.Q.bias: torch.Size([96])
  - decoder_stack.dec_layers.0.cross_attns.0.attn.Q.weight: torch.Size([96, 96])
  - decoder_stack.dec_layers.0.cross_attns.0.attn.V.bias: torch.Size([96])
  - decoder_stack.dec_layers.0.cross_attns.0.attn.V.weight: torch.Size([96, 96])
  - decoder_stack.dec_layer

In [6]:
converted_model_path = "./converted-models/prop"

In [9]:
# Convert baseline model to proposed architecture
import shutil

print("=" * 60)
print("CONVERTING BASELINE MODEL")
print("=" * 60)

# Create output directory
os.makedirs(converted_model_path, exist_ok=True)
print(f"\n✓ Created directory: {converted_model_path}")

# Read configs
with open(os.path.join(baseline_model_path, "config.json")) as f:
    baseline_config = json.load(f)
with open(os.path.join(proposed_model_path, "config.json")) as f:
    proposed_config = json.load(f)

# 1. Convert config
print("\n1. Converting config.json...")
converted_config = baseline_config.copy()

# Add keys that only exist in proposed model
only_in_proposed_cfg = set(proposed_config.keys()) - set(baseline_config.keys())
for key in only_in_proposed_cfg:
    converted_config[key] = proposed_config[key]
    print(f"   Added key '{key}' with value from proposed config")

# Overwrite vocab with proposed model's vocab
converted_config["vocab"] = proposed_config["vocab"]
print(f"   Overwritten key 'vocab' with value from proposed config")

del converted_config["merged_embedder"]["ap_embed"]

# Save converted config
config_output_path = os.path.join(converted_model_path, "config.json")
with open(config_output_path, 'w') as f:
    json.dump(converted_config, f, indent=2)
print(f"   ✓ Saved to {config_output_path}")

# Save the baseline model path into a text file for reference
baseline_path_output = os.path.join(converted_model_path, "baseline_model_path.txt")
with open(baseline_path_output, 'w') as f:
    f.write(baseline_model_path)
print(f"   ✓ Saved baseline model path to {baseline_path_output}")

# 2. Convert pytorch_model.bin
print("\n2. Converting pytorch_model.bin...")
baseline_state = torch.load(os.path.join(baseline_model_path, "pytorch_model.bin"), map_location='cpu')
converted_state = baseline_state.copy()

# Rename keys according to the mapping
rename_mapping = {
    "multi_head_enc_dec_attn": "cross_attns.0.attn",
    "norm_enc_dec_attn": "cross_attns.0.norm"
}

renamed_count = 0
for old_substring, new_substring in rename_mapping.items():
    # Find all keys containing the old substring and rename them
    keys_to_rename = [k for k in converted_state.keys() if old_substring in k]
    for old_key in keys_to_rename:
        new_key = old_key.replace(old_substring, new_substring)
        converted_state[new_key] = converted_state.pop(old_key)
        print(f"   Renamed: {old_key} → {new_key}")
        renamed_count += 1

print(f"   Total keys renamed: {renamed_count}")

# 3. Transform merged_embedder.base_weight
print("\n3. Transforming merged_embedder.base_weight...")
baseline_ap_count = len(baseline_config["vocab"]["aps"])
print(f"   baseline_ap_count: {baseline_ap_count}")

base_weight = converted_state["merged_embedder.base_weight"]
print(f"   Original shape: {base_weight.shape}")

# Extract parts
part1 = base_weight[baseline_ap_count:, :]
part2 = base_weight[:2, :]
print(f"   part1 shape (from index {baseline_ap_count}): {part1.shape}")
print(f"   part2 shape (first 2 rows): {part2.shape}")

# Concatenate
transformed_base_weight = torch.cat([part1, part2], dim=0)
print(f"   Transformed shape: {transformed_base_weight.shape}")

converted_state["merged_embedder.base_weight"] = transformed_base_weight

# Save converted state
model_output_path = os.path.join(converted_model_path, "pytorch_model.bin")
torch.save(converted_state, model_output_path)
print(f"   ✓ Saved to {model_output_path}")

print("\n✓ Conversion complete!")


CONVERTING BASELINE MODEL

✓ Created directory: ./converted-models/prop

1. Converting config.json...
   Added key 'cross_attn' with value from proposed config
   Added key 'no_enc_agg' with value from proposed config
   Added key 'no_dec_agg' with value from proposed config
   Overwritten key 'vocab' with value from proposed config
   ✓ Saved to ./converted-models/prop/config.json
   ✓ Saved baseline model path to ./converted-models/prop/baseline_model_path.txt

2. Converting pytorch_model.bin...
   Renamed: decoder_stack.dec_layers.0.multi_head_enc_dec_attn.Q.weight → decoder_stack.dec_layers.0.cross_attns.0.attn.Q.weight
   Renamed: decoder_stack.dec_layers.0.multi_head_enc_dec_attn.Q.bias → decoder_stack.dec_layers.0.cross_attns.0.attn.Q.bias
   Renamed: decoder_stack.dec_layers.0.multi_head_enc_dec_attn.K.weight → decoder_stack.dec_layers.0.cross_attns.0.attn.K.weight
   Renamed: decoder_stack.dec_layers.0.multi_head_enc_dec_attn.K.bias → decoder_stack.dec_layers.0.cross_attns.0.a

In [10]:
# Sanity check: compare converted model with proposed model
print("\n\nFINAL SANITY CHECK: Converted vs Proposed\n")
compare_models(converted_model_path, proposed_model_path, "Converted", "Proposed")




FINAL SANITY CHECK: Converted vs Proposed

CONFIG COMPARISON

✓ No keys only in Converted config

✗ No keys only in Proposed config

✓ Common keys (20): ['cross_attn', 'd_embed_dec', 'd_embed_enc', 'd_ff', 'datatype', 'dec_pe', 'dropout', 'enc_pe', 'ff_activation', 'layer_norm_eps', 'max_decode_length', 'max_encode_length', 'merged_embedder', 'no_dec_agg', 'no_enc_agg', 'no_pe_cross_keys', 'num_heads', 'num_layers', 'tree_pos_enc', 'vocab']

PYTORCH MODEL PARAMETER COMPARISON

✓ No parameters only in Converted model

✗ No parameters only in Proposed model

✓ Common parameters (253):
  - decoder_stack.dec_layers.5.cross_attns.0.attn.K.bias: torch.Size([132])
  - decoder_stack.dec_layers.5.multi_head_self_attn.Q.weight: torch.Size([132, 132])
  - encoder_stack.enc_layers.2.multi_head_attn.V.bias: torch.Size([132])
  - encoder_stack.enc_layers.3.norm_attn.weight: torch.Size([132])
  - encoder_stack.enc_layers.5.norm_ff.bias: torch.Size([132])
  ... and 248 more

MERGED_EMBEDDER.BASE_WEI

**NOTE:** Remove shuffle_aps key manually from the converted model.

In [ ]:
# Load pytorch models
state1 = torch.load(os.path.join(proposed_model_path, "pytorch_model.bin"), map_location='cpu')
state2 = torch.load(os.path.join(converted_model_path, "pytorch_model.bin"), map_location='cpu')

In [21]:
state1["merged_embedder.base_weight"].size()

torch.Size([15, 64])

In [22]:
state2["merged_embedder.base_weight"].size()

torch.Size([23, 128])